# Benchmark методов поиска факторов — v9 / Tournament Safe

Версия для большого датасета и нестабильного kernel.

Архитектура:
1. **Tournament:** все 6 методов, 1 seed, 150k TRAIN + 50k VALID, до 30 CatBoost-fit.
2. **Confirmation:** автоматически выбираются top-3 метода; они повторяются на 300k/100k по 3 seed.
3. **FULL VALID:** лучшие уникальные наборы финалистов переоцениваются на полном TRAIN/VALID.
4. **OOS:** только лучший FULL-набор каждого метода проверяется на OOS.

Дополнительно:
- `thread_count=8`;
- heartbeat после каждого fit;
- fit-level журнал на диск;
- method×seed checkpoint и `resume=True`;
- CatBoost-модели удаляются сразу после каждого fit;
- light baseline по умолчанию отключён;
- все Greedy / Beam / Random / TPE / GP / CMA-ES обязательно участвуют в Tournament.


## Изменения протокола v2

Эта версия предназначена для честного сравнения методов поиска состава факторов.

Ключевые правила:

- **OOS не участвует** в отборе факторов, preprocessing, stopping и выборе гиперпараметров.
- Финальный набор определяется только по TRAIN/VALID.
- Сохраняется лучший найденный набор, а не последний набор перед остановкой.
- Ограничение `min_features_before_stop` относится к фактическому числу факторов.
- Все `model.fit()` учитываются в вычислительном бюджете.
- Отдельно измеряются время поиска, финального обучения и полное время.
- Baseline на всех факторах должен запускаться на тех же seed, что и методы поиска.
- Автоматически категориальными считаются только `object`, `category` и `bool`; числовые low-cardinality признаки не переводятся в категории без явного разрешения.
- Drift-диагностика должна считаться на исходных данных до технических преобразований CatBoost.
- Для сравнения количества факторов сохраняется frontier: `N факторов → лучший найденный VALID Gini`.
- OOS используется только для однократной финальной оценки выбранного решения каждого независимого запуска.

> Важно: VALID всё ещё используется как eval-set CatBoost и как критерий feature search. Для строгого исследовательского режима рекомендуется отдельный внутренний eval/CV внутри TRAIN; OOS при этом остаётся полностью закрытым до финала.


In [ ]:
# Базовые зависимости:
# %pip install -U pandas numpy scikit-learn catboost psutil openpyxl
#
# Для TPE / GP / CMA-ES:
# %pip install -U "optuna>=4.4" cmaes scipy

import gc
import json
import os
import sys
import time
import warnings
import importlib
import subprocess
from dataclasses import dataclass, field, asdict, replace
from itertools import combinations
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

try:
    import psutil
except Exception:
    psutil = None

pd.set_option("display.max_columns", 200)


def _import_optional_search_dependencies():
    """Импортирует зависимости оптимизаторов и возвращает диагностический отчёт."""
    report = {}

    try:
        import optuna as _optuna
        _optuna.logging.set_verbosity(_optuna.logging.WARNING)
        report["optuna"] = getattr(_optuna, "__version__", "unknown")
    except Exception as e:
        _optuna = None
        report["optuna"] = f"MISSING: {e!r}"

    try:
        import cmaes as _cmaes
        report["cmaes"] = getattr(_cmaes, "__version__", "installed")
    except Exception as e:
        _cmaes = None
        report["cmaes"] = f"MISSING: {e!r}"

    try:
        import scipy as _scipy
        report["scipy"] = getattr(_scipy, "__version__", "installed")
    except Exception as e:
        _scipy = None
        report["scipy"] = f"MISSING: {e!r}"

    return _optuna, report


optuna, SEARCH_DEPENDENCY_REPORT = _import_optional_search_dependencies()
print("Search dependencies:", SEARCH_DEPENDENCY_REPORT)


## 1. Настройки

Ниже находится **единственный конфиг всего benchmark**.

Менять параметры нужно только в `class Config`: пути к данным, target, методы,
seed, размеры выборок, бюджеты, CatBoost, ресурсы и выходную директорию.

Никакого второго `Config(...)` с переопределениями ниже больше нет.


In [ ]:
@dataclass
class Config:
    train: str = "train.parquet"
    valid: str = "valid.parquet"
    oos: str = "oos.parquet"
    target: str = "target"

    id_columns: List[str] = field(default_factory=list)
    exclude_columns: List[str] = field(default_factory=list)
    categorical_columns: List[str] = field(default_factory=list)
    datetime_columns: List[str] = field(default_factory=list)

    auto_categorical: bool = True
    auto_integer_categorical: bool = False
    max_auto_cat_unique: int = 50
    auto_datetime: bool = True
    datetime_parse_threshold: float = 0.95
    datetime_features: List[str] = field(
        default_factory=lambda: ["timestamp", "year", "month", "day", "dayofweek"]
    )

    # ---------- РЕСУРСЫ ----------
    # Не используем все ядра: CatBoost на -1 способен резко поднять RAM.
    thread_count: int = 8

    # Мягкая остановка ДО OOM. Проверяется перед каждым fit.
    max_ram_percent: float = 72.0

    # Числовые float64 по возможности переводятся в float32.
    downcast_float64: bool = True

    # Drift тяжёлый и не нужен для самого benchmark.
    calculate_drift: bool = False

    # ---------- LARGE-DATA SEARCH SAMPLE ----------
    search_train_rows: int = 300_000
    search_valid_rows: int = 100_000
    search_sample_seed: int = 2026
    search_iterations: int = 140

    # ---------- ОБЩИЙ SCREENING ----------
    # Один лёгкий CatBoost видит все признаки одновременно.
    # Берём top importance + случайную exploration-добавку.
    screening_enabled: bool = True
    screening_top_n: int = 220
    screening_random_n: int = 50
    screening_iterations: int = 180

    # ---------- ДВУХЭТАПНЫЙ LARGE-DATA ПРОТОКОЛ ----------
    tournament_train_rows: int = 150_000
    tournament_valid_rows: int = 50_000
    tournament_fits: int = 30
    tournament_iterations: int = 90
    tournament_seed: int = 42

    confirmation_train_rows: int = 300_000
    confirmation_valid_rows: int = 100_000
    confirmation_fits: int = 30
    confirmation_iterations: int = 120
    confirmation_top_methods: int = 3
    confirmation_seeds: List[int] = field(default_factory=lambda: [42, 2026, 3407])

    # FULL: максимум столько уникальных наборов на метод после confirmation.
    full_candidates_per_method: int = 3

    # Печать состояния и журнал после каждого fit.
    heartbeat_every_fit: int = 1
    resume: bool = True

    # ---------- СРАВНЕНИЕ МЕТОДОВ ----------
    methods: List[str] = field(
        default_factory=lambda: ["greedy", "beam", "random", "tpe", "gp", "cmaes"]
    )

    # Первый проход — один seed. После выбора финалистов можно поставить 3.
    repeats: int = 1
    seeds: List[int] = field(default_factory=lambda: [42])

    start_features: List[str] = field(default_factory=list)
    max_features: int = 200
    delta_gini: float = 0.001
    plateau_patience: int = 3
    min_features_before_stop: int = 80
    additions_per_step: int = 20

    # Сколько альтернатив рассматривается на одном шаге.
    candidate_pool_size: int = 6
    beam_width: int = 2
    random_trials_per_step: int = 12
    optimizer_trials_per_step: int = 60
    optimizer_timeout_per_step: Optional[int] = None
    # Компактное пространство TPE/GP/CMA-ES:
    optimizer_block_size: int = 10
    optimizer_max_blocks: int = 20
    optimizer_min_blocks: int = 2


    # ЕДИНЫЙ бюджет поиска на method × seed.
    max_model_fits: int = 30
    max_wall_seconds: int = 1800

    # ---------- LIGHT MODEL ДЛЯ SEARCH ----------
    search_catboost_params: Dict[str, Any] = field(default_factory=lambda: {
        "iterations": 220,
        "depth": 6,
        "learning_rate": 0.07,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "verbose": False,
        "allow_writing_files": False,
    })
    search_early_stopping_rounds: int = 35

    # ---------- ПОЛНАЯ ПЕРЕОЦЕНКА ----------
    final_catboost_params: Dict[str, Any] = field(default_factory=lambda: {
        "iterations": 700,
        "depth": 6,
        "learning_rate": 0.05,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "verbose": False,
        "allow_writing_files": False,
    })
    final_early_stopping_rounds: int = 80

    # Сколько лучших УНИКАЛЬНЫХ наборов из search-траектории
    # переоценить полноценной моделью на VALID.
    top_k_full_evaluation: int = 3

    catboost_ram_limit: str = "100gb"
    run_light_baseline: bool = False
    run_full_baseline_at_end: bool = False
    baseline_iterations: int = 120
    baseline_early_stopping_rounds: int = 20
    output_dir: str = "feature_benchmark_v9_tournament_safe"
    save_models: bool = False

# Единственный объект конфигурации.
# Все параметры запуска меняются ТОЛЬКО в class Config выше.
cfg = Config()
cfg


In [ ]:
def validate_experiment_protocol(cfg: Config):
    if cfg.repeats != len(cfg.seeds):
        raise ValueError(
            f"repeats={cfg.repeats}, seeds={len(cfg.seeds)}. "
            "Для воспроизводимого сравнения они должны совпадать."
        )
    if cfg.additions_per_step < 1:
        raise ValueError("additions_per_step должен быть >= 1")
    if cfg.max_features < 1:
        raise ValueError("max_features должен быть >= 1")
    if cfg.max_model_fits < 1:
        raise ValueError("max_model_fits должен быть >= 1")
    if not (20 <= cfg.max_ram_percent <= 95):
        raise ValueError("max_ram_percent разумно задавать в диапазоне 20..95")
    if cfg.top_k_full_evaluation < 1:
        raise ValueError("top_k_full_evaluation должен быть >= 1")
    print("Конфигурация прошла базовую проверку.")

validate_experiment_protocol(cfg)


def validate_search_dependencies(cfg: Config):
    """
    Проверка выполняется ДО загрузки/benchmark.
    Если заявлены TPE/GP/CMA-ES, notebook не имеет права молча их пропустить.
    """
    requested = set(cfg.methods)
    optimizer_methods = requested & {"tpe", "gp", "cmaes"}
    if not optimizer_methods:
        return

    problems = []

    if optuna is None:
        problems.append("не установлен optuna>=4.4")
    else:
        if "gp" in optimizer_methods and not hasattr(optuna.samplers, "GPSampler"):
            problems.append(
                f"Optuna {getattr(optuna, '__version__', '?')} не содержит GPSampler"
            )

    if "cmaes" in optimizer_methods:
        try:
            import cmaes  # noqa: F401
        except Exception:
            problems.append("для CMA-ES не установлен пакет cmaes")

    if "gp" in optimizer_methods:
        try:
            import scipy  # noqa: F401
        except Exception:
            problems.append("для GP не установлен scipy")

    if problems:
        raise ImportError(
            "Не пройден preflight методов поиска:\n- "
            + "\n- ".join(problems)
            + '\n\nУстановите зависимости в отдельной ячейке:\n'
              '%pip install -U "optuna>=4.4" cmaes scipy\n'
              "Затем выполните Restart Kernel и запустите notebook заново."
        )

    print(
        "Preflight методов: OK | "
        f"methods={cfg.methods} | "
        f"optuna={getattr(optuna, '__version__', '?')}"
    )


## 2. Загрузка, аудит типов и минимальный preprocessing

Перед первым обучением ноутбук строит `type_audit.csv`.

### Обработка дат в v4

Автоопределение дат теперь двухступенчатое и выполняется **только по TRAIN**:

1. Быстрый regex-фильтр проверяет небольшой sample строк и отбрасывает обычные текстовые/категориальные факторы без попытки парсинга.
2. Только колонки, действительно похожие на даты, проверяются векторным `pd.to_datetime()` с явными форматами / `format="mixed"`.

Это устраняет поток `Could not infer format...`, не запускает дорогой `dateutil` для каждой строковой колонки и заметно ускоряет preprocessing на широких витринах.

Настоящий `datetime64` определяется напрямую. После распознавания исходная дата заменяется на числовые компоненты (`timestamp`, `year`, `month`, `day`, `dayofweek`). VALID и OOS получают ту же TRAIN-схему и не участвуют в выборе преобразования.

Сложные объектные поля (`list`, `dict`, массивы и т. п.) останавливают запуск до первого CatBoost fit с понятным аудитом типов.


In [ ]:
def load_table(path: str) -> pd.DataFrame:
    """
    Загружает таблицу из одиночного файла или директории с Parquet-part'ами.
    """
    p = Path(path).expanduser()
    if not p.exists():
        raise FileNotFoundError(f"Путь не найден: {p}")

    if p.is_dir():
        try:
            return pd.read_parquet(p)
        except Exception as directory_error:
            parquet_files = sorted(
                f for f in p.rglob("*")
                if f.is_file() and f.suffix.lower() in {".parquet", ".pq"}
            )
            if not parquet_files:
                raise ValueError(
                    f"В директории {p} не найдено Parquet-файлов."
                ) from directory_error
            return pd.concat(
                [pd.read_parquet(f) for f in parquet_files],
                ignore_index=True,
            )

    ext = p.suffix.lower()
    if ext in {".parquet", ".pq"}:
        return pd.read_parquet(p)
    if ext in {".csv", ".txt"}:
        return pd.read_csv(p)
    if ext in {".pkl", ".pickle"}:
        return pd.read_pickle(p)
    if ext in {".xlsx", ".xls"}:
        return pd.read_excel(p)
    if ext == ".feather":
        return pd.read_feather(p)

    raise ValueError(f"Неподдерживаемый формат файла: {p.suffix}")


def validate_binary_target(y: pd.Series, name: str) -> pd.Series:
    """Проверяет бинарный target и при необходимости детерминированно переводит его в 0/1."""
    if y.isna().any():
        raise ValueError(f"{name}: в target есть пропуски")
    vals = list(pd.unique(y))
    if len(vals) != 2:
        raise ValueError(f"{name}: нужен бинарный target, получены значения {vals[:10]}")
    if set(vals) == {0, 1} or set(vals) == {False, True}:
        return y.astype(int)
    mapping = {v: i for i, v in enumerate(sorted(vals, key=lambda x: str(x)))}
    warnings.warn(f"{name}: target преобразован в 0/1 по правилу {mapping}")
    return y.map(mapping).astype(int)


def _dtype_name(s: pd.Series) -> str:
    return str(s.dtype)


def _has_complex_python_objects(s: pd.Series, sample_size: int = 1000) -> bool:
    """
    Ищет значения, которые CatBoost не должен получать напрямую:
    list/dict/set/tuple/ndarray и другие вложенные структуры.
    """
    x = s.dropna().head(sample_size)
    complex_types = (list, dict, set, tuple, np.ndarray)
    return any(isinstance(v, complex_types) for v in x)


def _looks_like_datetime_strings(
    s: pd.Series,
    sample_size: int = 500,
    min_pattern_rate: float = 0.80,
) -> Tuple[bool, float]:
    """
    Дешёвый предварительный фильтр для строковых дат.

    Сначала проверяем только форму строк и НЕ запускаем dateutil.
    В парсинг попадают лишь колонки, где большинство непустых значений
    похоже на дату/дату-время: YYYY-MM-DD, YYYY/MM/DD, DD.MM.YYYY,
    ISO datetime с T, дата со временем и т. п.

    Решение принимается только по TRAIN.
    """
    x = s.dropna()
    if x.empty:
        return False, 0.0

    if len(x) > sample_size:
        x = x.sample(sample_size, random_state=42)

    x = x.astype("string").str.strip()

    patterns = [
        r"^\d{4}[-/]\d{1,2}[-/]\d{1,2}(?:[ T].*)?$",
        r"^\d{1,2}[./-]\d{1,2}[./-]\d{4}(?:[ T].*)?$",
        r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}",
        r"^\d{4}\d{2}\d{2}$",
    ]

    mask = False
    for pattern in patterns:
        current = x.str.match(pattern, na=False)
        mask = current if isinstance(mask, bool) else (mask | current)

    rate = float(mask.mean())
    return rate >= min_pattern_rate, rate


def _datetime_parse_rate(
    s: pd.Series,
    sample_size: int = 500,
) -> Tuple[float, Optional[str]]:
    """
    Подтверждает дату только после дешёвого regex-фильтра.

    Сначала пробуем несколько распространённых форматов в векторном режиме.
    Только если они не подходят, используем format='mixed' (Pandas >= 2),
    также без предупреждений о невозможности вывести единый формат.

    Возвращает:
      parse_rate — долю успешно распарсенных значений;
      format_hint — найденный формат либо 'mixed'.
    """
    x = s.dropna()
    if x.empty:
        return 0.0, None

    if len(x) > sample_size:
        x = x.sample(sample_size, random_state=42)

    x = x.astype("string").str.strip()

    formats = [
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%d.%m.%Y",
        "%d-%m-%Y",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%dT%H:%M:%S.%f",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M:%S.%f",
        "%Y%m%d",
    ]

    best_rate = 0.0
    best_format = None

    for fmt in formats:
        parsed = pd.to_datetime(x, format=fmt, errors="coerce")
        rate = float(parsed.notna().mean())
        if rate > best_rate:
            best_rate = rate
            best_format = fmt
        if rate >= 0.995:
            return rate, fmt

    # Для смешанных, но явно похожих на даты значений.
    try:
        parsed = pd.to_datetime(x, format="mixed", errors="coerce")
        mixed_rate = float(parsed.notna().mean())
        if mixed_rate > best_rate:
            return mixed_rate, "mixed"
    except (TypeError, ValueError):
        # Совместимость со старыми версиями pandas:
        # не запускаем медленный неявный dateutil-парсинг.
        pass

    return best_rate, best_format


def detect_datetime_columns(
    train: pd.DataFrame,
    features: Sequence[str],
    cfg: Config,
) -> Tuple[List[str], Dict[str, str]]:
    """
    Определяет даты только по TRAIN.
    Явно заданные datetime_columns имеют приоритет.
    Автоопределение применяется к настоящим datetime dtype и строковым/object полям.
    Чистые числовые признаки автоматически датами не считаются.
    """
    detected = []
    reason = {}
    explicit = set(cfg.datetime_columns)

    for c in features:
        s = train[c]

        if c in explicit:
            detected.append(c)
            reason[c] = "DATETIME_EXPLICIT"
            continue

        if pd.api.types.is_datetime64_any_dtype(s):
            detected.append(c)
            reason[c] = "DATETIME_DTYPE"
            continue

        if not cfg.auto_datetime:
            continue

        if (
            pd.api.types.is_object_dtype(s)
            or pd.api.types.is_string_dtype(s)
        ):
            looks_like_date, pattern_rate = _looks_like_datetime_strings(s)

            # Обычные текстовые/категориальные поля дальше не парсим вообще.
            if not looks_like_date:
                continue

            rate, format_hint = _datetime_parse_rate(s)
            if rate >= cfg.datetime_parse_threshold:
                detected.append(c)
                reason[c] = (
                    f"DATETIME_AUTO_parse={rate:.3f};"
                    f"pattern={pattern_rate:.3f};format={format_hint}"
                )

    return detected, reason


def expand_datetime_columns(
    train: pd.DataFrame,
    valid: pd.DataFrame,
    oos: pd.DataFrame,
    datetime_cols: Sequence[str],
    cfg: Config,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, List[str]]]:
    """
    Преобразует дату в числовые признаки.
    Схема преобразования определяется заранее и одинакова для TRAIN/VALID/OOS.
    OOS не участвует в решении, какие даты или компоненты использовать.
    """
    # Resource-safe: преобразуем переданные DataFrame на месте.
    # Полные копии широких TRAIN/VALID/OOS здесь не создаются.
    generated = {}

    for c in datetime_cols:
        new_cols = []

        for df_name, df in [("TRAIN", train), ("VALID", valid), ("OOS", oos)]:
            # Явный/mixed формат не даёт Pandas перебирать dateutil
            # для каждой строки и не создаёт поток UserWarning.
            try:
                parsed = pd.to_datetime(df[c], format="mixed", errors="coerce")
            except (TypeError, ValueError):
                # Для старых Pandas используем ISO8601; явно заданные нестандартные
                # даты при необходимости лучше перечислить в datetime_columns
                # и предварительно привести к единому формату.
                parsed = pd.to_datetime(df[c], format="ISO8601", errors="coerce")

            # Если явно/автоматически признанная дата вообще не парсится на TRAIN,
            # это ошибка схемы, а не повод молча менять решение по VALID/OOS.
            if df_name == "TRAIN" and parsed.notna().sum() == 0 and df[c].notna().sum() > 0:
                raise ValueError(
                    f"Колонка '{c}' определена как дата, но значения TRAIN не удалось распарсить."
                )

            if "timestamp" in cfg.datetime_features:
                # Unix timestamp в секундах. NaT -> NaN.
                values = parsed.astype("int64", copy=False).astype("float64") / 1e9
                values[parsed.isna()] = np.nan
                df[f"{c}__timestamp"] = values

            if "year" in cfg.datetime_features:
                df[f"{c}__year"] = parsed.dt.year.astype("float64")
            if "month" in cfg.datetime_features:
                df[f"{c}__month"] = parsed.dt.month.astype("float64")
            if "day" in cfg.datetime_features:
                df[f"{c}__day"] = parsed.dt.day.astype("float64")
            if "dayofweek" in cfg.datetime_features:
                df[f"{c}__dayofweek"] = parsed.dt.dayofweek.astype("float64")
            if "quarter" in cfg.datetime_features:
                df[f"{c}__quarter"] = parsed.dt.quarter.astype("float64")

        for suffix in cfg.datetime_features:
            name = f"{c}__{suffix}"
            if name in train.columns:
                new_cols.append(name)

        generated[c] = new_cols

        # Исходная дата в CatBoost больше не передаётся.
        train.drop(columns=[c], inplace=True)
        valid.drop(columns=[c], inplace=True)
        oos.drop(columns=[c], inplace=True)

    return train, valid, oos, generated


def build_type_audit(
    train_raw: pd.DataFrame,
    valid_raw: pd.DataFrame,
    oos_raw: pd.DataFrame,
    candidates: Sequence[str],
    datetime_cols: Sequence[str],
    datetime_reason: Dict[str, str],
) -> pd.DataFrame:
    """Формирует понятный аудит типов до запуска первого CatBoost fit."""
    rows = []
    dt_set = set(datetime_cols)

    for c in candidates:
        action = "KEEP"
        note = ""

        if c in dt_set:
            action = "EXPAND_DATETIME"
            note = datetime_reason.get(c, "")
        elif _has_complex_python_objects(train_raw[c]):
            action = "ERROR_COMPLEX_OBJECT"
            note = "В колонке есть list/dict/array/tuple/set"
        elif pd.api.types.is_numeric_dtype(train_raw[c]):
            action = "NUMERIC"
        elif (
            pd.api.types.is_object_dtype(train_raw[c])
            or pd.api.types.is_string_dtype(train_raw[c])
            or isinstance(train_raw[c].dtype, pd.CategoricalDtype)
            or pd.api.types.is_bool_dtype(train_raw[c])
        ):
            action = "CATEGORICAL"
        else:
            action = "UNSUPPORTED_DTYPE"
            note = f"Необработанный dtype TRAIN: {train_raw[c].dtype}"

        rows.append({
            "feature": c,
            "dtype_train": _dtype_name(train_raw[c]),
            "dtype_valid": _dtype_name(valid_raw[c]),
            "dtype_oos": _dtype_name(oos_raw[c]),
            "action": action,
            "note": note,
        })

    return pd.DataFrame(rows)


def infer_categoricals(
    train: pd.DataFrame,
    cfg: Config,
    features: Sequence[str],
) -> List[str]:
    """
    Категориальность определяется по TRAIN.
    По умолчанию integer с малым числом значений НЕ считается категорией автоматически.
    """
    explicit = [c for c in cfg.categorical_columns if c in features]
    if not cfg.auto_categorical:
        return sorted(set(explicit))

    cats = set(explicit)
    for c in features:
        s = train[c]
        if (
            pd.api.types.is_object_dtype(s)
            or pd.api.types.is_string_dtype(s)
            or isinstance(s.dtype, pd.CategoricalDtype)
            or pd.api.types.is_bool_dtype(s)
        ):
            cats.add(c)
        elif (
            cfg.auto_integer_categorical
            and pd.api.types.is_integer_dtype(s)
            and s.nunique(dropna=True) <= cfg.max_auto_cat_unique
        ):
            cats.add(c)
    return sorted(cats)


def downcast_numeric_inplace(df: pd.DataFrame, features: Sequence[str], enabled: bool = True):
    """Экономит RAM: float64 -> float32, безопасные int downcast без создания полной копии DataFrame."""
    if not enabled:
        return
    for c in features:
        if c not in df.columns:
            continue
        s = df[c]
        if pd.api.types.is_float_dtype(s) and s.dtype == np.float64:
            df[c] = pd.to_numeric(s, downcast="float")
        elif pd.api.types.is_integer_dtype(s):
            # target сюда не передаётся; downcast только факторов.
            df[c] = pd.to_numeric(s, downcast="integer")


def stratified_binary_sample(df, target, n_rows, seed):
    """Стратифицированный sample с сохранением доли target; полный df не копируется."""
    if n_rows is None or n_rows <= 0 or n_rows >= len(df):
        return df
    rng = np.random.default_rng(seed)
    y = df[target].to_numpy()
    classes, counts = np.unique(y, return_counts=True)
    proportions = counts / counts.sum()
    parts, remaining = [], int(n_rows)
    for i, (cls, prop) in enumerate(zip(classes, proportions)):
        idx = np.flatnonzero(y == cls)
        take = min(remaining if i == len(classes)-1 else int(round(n_rows*prop)), len(idx))
        if i < len(classes)-1:
            remaining -= take
        parts.append(rng.choice(idx, size=take, replace=False))
    chosen = np.concatenate(parts)
    rng.shuffle(chosen)
    return df.iloc[chosen]


def technical_filter(
    train: pd.DataFrame,
    valid: pd.DataFrame,
    oos: pd.DataFrame,
    cfg: Config,
):
    """
    Технический preprocessing без target-based отбора.

    Важно:
    - решения о типах и удалении факторов принимаются только по TRAIN;
    - VALID используется для обучения/выбора позже;
    - OOS здесь только преобразуется по уже принятой TRAIN-схеме;
    - OOS никогда не определяет, оставить или удалить фактор.
    """
    for name, df in [("TRAIN", train), ("VALID", valid), ("OOS", oos)]:
        if cfg.target not in df.columns:
            raise ValueError(f"{name}: отсутствует target '{cfg.target}'")

    # Схема признаков определяется по TRAIN.
    missing_valid = [c for c in train.columns if c not in valid.columns and c != cfg.target]
    missing_oos = [c for c in train.columns if c not in oos.columns and c != cfg.target]
    if missing_valid:
        raise ValueError(f"VALID: отсутствуют колонки TRAIN: {missing_valid[:20]}")
    if missing_oos:
        raise ValueError(f"OOS: отсутствуют колонки TRAIN: {missing_oos[:20]}")

    excluded = set([cfg.target] + cfg.id_columns + cfg.exclude_columns)
    candidates = [c for c in train.columns if c not in excluded]

    # Сначала аудитируем исходные типы.
    datetime_cols, datetime_reason = detect_datetime_columns(train, candidates, cfg)
    type_audit = build_type_audit(
        train, valid, oos, candidates, datetime_cols, datetime_reason
    )

    fatal = type_audit[type_audit["action"].isin(
        ["ERROR_COMPLEX_OBJECT", "UNSUPPORTED_DTYPE"]
    )]
    if not fatal.empty:
        display(fatal.head(50))
        raise TypeError(
            "Обнаружены факторы с неподдерживаемыми типами. "
            "Смотрите таблицу type_audit выше; такие поля нужно исключить "
            "или явно преобразовать до benchmark."
        )

    # Константы удаляем ТОЛЬКО по TRAIN.
    technical_rows = []
    base_keep = []
    for c in candidates:
        if train[c].nunique(dropna=False) <= 1:
            technical_rows.append({
                "feature": c,
                "status": "DROP_CONSTANT_TRAIN",
                "details": "Константа на TRAIN",
            })
        else:
            base_keep.append(c)
            technical_rows.append({
                "feature": c,
                "status": "KEEP_OR_TRANSFORM",
                "details": datetime_reason.get(c, ""),
            })

    datetime_cols = [c for c in datetime_cols if c in base_keep]

    # Даты раскладываем на числовые компоненты одинаково во всех трёх срезах.
    train_p, valid_p, oos_p, generated_dates = expand_datetime_columns(
        train, valid, oos, datetime_cols, cfg
    )

    # Итоговый список факторов: обычные + производные от дат.
    keep = [c for c in base_keep if c not in datetime_cols]
    for original in datetime_cols:
        keep.extend(generated_dates.get(original, []))

    # +/-inf заменяем на NaN только в числовых полях.
    for df in (train_p, valid_p, oos_p):
        numeric = [c for c in keep if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
        if numeric:
            df.loc[:, numeric] = df[numeric].replace([np.inf, -np.inf], np.nan)

    cats = infer_categoricals(train_p, cfg, keep)

    # CatBoost не принимает NaN как значение категориального признака.
    # Приведение схемы выполняется одинаково, но решение о cats принято только по TRAIN.
    for df in (train_p, valid_p, oos_p):
        for c in cats:
            df[c] = df[c].astype("string").fillna("__MISSING__")

    # Финальная защита: в числовые факторы не должны просочиться строки/datetime.
    bad_numeric = []
    for c in keep:
        if c in cats:
            continue
        if not pd.api.types.is_numeric_dtype(train_p[c]):
            bad_numeric.append((c, str(train_p[c].dtype)))

    if bad_numeric:
        raise TypeError(
            "После preprocessing остались нечисловые факторы, не объявленные категориальными: "
            + repr(bad_numeric[:20])
        )

    # Уменьшаем размер числовой матрицы после всех преобразований.
    for df in (train_p, valid_p, oos_p):
        downcast_numeric_inplace(df, [c for c in keep if c not in cats], cfg.downcast_float64)

    technical = pd.DataFrame(technical_rows)
    gc.collect()
    return train_p, valid_p, oos_p, keep, cats, technical, type_audit, generated_dates


## 3. Gini и единый контур обучения CatBoost

In [ ]:
def gini(y_true: Sequence[int], p: Sequence[float]) -> float:
    return 2.0 * roc_auc_score(y_true, p) - 1.0


class BudgetExceeded(RuntimeError):
    pass


class ResourceLimit(RuntimeError):
    pass


def ram_status() -> Dict[str, float]:
    if psutil is None:
        return {"percent": np.nan, "available_gb": np.nan}
    vm = psutil.virtual_memory()
    return {
        "percent": float(vm.percent),
        "available_gb": float(vm.available / (1024 ** 3)),
    }


def print_ram(prefix: str = ""):
    r = ram_status()
    if np.isfinite(r["percent"]):
        print(f"{prefix}RAM: {r['percent']:.1f}% | доступно {r['available_gb']:.2f} GB")


class Evaluator:
    """
    Search-evaluator использует только облегчённый CatBoost.
    Полноценная модель вызывается отдельно только для top-K наборов.
    """
    def __init__(self, train, valid, target, categorical, cfg, method, repeat, seed):
        self.train = train
        self.valid = valid
        self.target = target
        self.categorical = set(categorical)
        self.cfg = cfg
        self.method = method
        self.repeat = repeat
        self.seed = seed
        self.started = time.perf_counter()
        self.fit_count = 0
        self.cache = {}
        self.rows = []

    def _check_budget(self):
        if self.fit_count >= self.cfg.max_model_fits:
            raise BudgetExceeded("Достигнут max_model_fits")
        if (time.perf_counter() - self.started) >= self.cfg.max_wall_seconds:
            raise BudgetExceeded("Достигнут max_wall_seconds")
        r = ram_status()
        if np.isfinite(r["percent"]) and r["percent"] >= self.cfg.max_ram_percent:
            gc.collect()
            r2 = ram_status()
            if r2["percent"] >= self.cfg.max_ram_percent:
                raise ResourceLimit(
                    f"RAM {r2['percent']:.1f}% >= лимита {self.cfg.max_ram_percent:.1f}%"
                )

    def evaluate(self, features: Sequence[str], stage: str, note: str = "") -> Dict[str, Any]:
        fs = tuple(sorted(set(features)))
        if not fs:
            raise ValueError("Нельзя обучить модель без факторов")
        if fs in self.cache:
            out = dict(self.cache[fs])
            out["cached"] = True
            return out

        self._check_budget()
        cats = [c for c in fs if c in self.categorical]
        params = dict(self.cfg.search_catboost_params)
        params["iterations"] = self.cfg.search_iterations
        params["random_seed"] = self.seed
        params["thread_count"] = self.cfg.thread_count
        params["used_ram_limit"] = self.cfg.catboost_ram_limit

        model = None
        t0 = time.perf_counter()
        try:
            model = CatBoostClassifier(**params)
            model.fit(
                self.train.loc[:, list(fs)],
                self.train[self.target],
                cat_features=cats,
                eval_set=(self.valid.loc[:, list(fs)], self.valid[self.target]),
                early_stopping_rounds=self.cfg.search_early_stopping_rounds,
                verbose=False,
            )
            pv = model.predict_proba(self.valid.loc[:, list(fs)])[:, 1]
            gv = gini(self.valid[self.target], pv)
            best_iter = int(model.get_best_iteration())
        finally:
            elapsed = time.perf_counter() - t0
            # Освобождаем нативные структуры CatBoost сразу после evaluation.
            if model is not None:
                del model
            gc.collect()

        self.fit_count += 1
        result = {
            "features": list(fs),
            "n_features": len(fs),
            "valid_gini": gv,
            "fit_seconds": elapsed,
            "best_iteration": best_iter,
            "cached": False,
        }
        self.cache[fs] = dict(result)
        row = {
            "method": self.method,
            "repeat": self.repeat,
            "seed": self.seed,
            "fit_no": self.fit_count,
            "stage": stage,
            "note": note,
            "n_features": len(fs),
            "valid_gini": gv,
            "fit_seconds": elapsed,
            "elapsed_total": time.perf_counter() - self.started,
            "best_iteration": best_iter,
            "features": json.dumps(list(fs), ensure_ascii=False),
        }
        self.rows.append(row)

        # Fit-level журнал переживает падение kernel.
        try:
            journal = Path(self.cfg.output_dir) / "fit_journal.jsonl"
            with journal.open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(row, ensure_ascii=False) + "\n")
        except Exception as e:
            warnings.warn(f"Не удалось записать fit_journal: {e}")

        if self.fit_count % max(1, self.cfg.heartbeat_every_fit) == 0:
            rs = ram_status()
            print(
                f"[{self.method} | seed={self.seed}] "
                f"fit {self.fit_count}/{self.cfg.max_model_fits} | "
                f"N={len(fs)} | Gini={gv:.5f} | "
                f"fit={elapsed:.1f}s | "
                f"elapsed={(time.perf_counter()-self.started)/60:.1f}m | "
                f"RAM={rs['percent']:.1f}%"
            )

        # Prediction больше не нужен.
        try:
            del pv
        except Exception:
            pass
        gc.collect()
        return result


def fit_full_on_valid(
    train, valid, target, cats_all, features, cfg: Config, seed: int
) -> Dict[str, Any]:
    """Полноценная TRAIN→VALID переоценка одного shortlist-кандидата."""
    fs = sorted(set(features))
    cats = [c for c in fs if c in set(cats_all)]
    params = dict(cfg.final_catboost_params)
    params["random_seed"] = seed
    params["thread_count"] = cfg.thread_count
    params["used_ram_limit"] = cfg.catboost_ram_limit

    model = CatBoostClassifier(**params)
    t0 = time.perf_counter()
    try:
        model.fit(
            train.loc[:, fs], train[target],
            cat_features=cats,
            eval_set=(valid.loc[:, fs], valid[target]),
            early_stopping_rounds=cfg.final_early_stopping_rounds,
            verbose=False,
        )
        pv = model.predict_proba(valid.loc[:, fs])[:, 1]
        return {
            "features": fs,
            "n_features": len(fs),
            "full_valid_gini": gini(valid[target], pv),
            "best_iteration": int(model.get_best_iteration()),
            "full_fit_seconds": time.perf_counter() - t0,
        }
    finally:
        del model
        gc.collect()


def light_all_features_baseline(train, valid, target, cats_all, features, cfg, seed):
    """Light TRAIN→VALID baseline на всех факторах. OOS здесь не используется."""
    r = ram_status()
    if np.isfinite(r["percent"]) and r["percent"] >= cfg.max_ram_percent:
        gc.collect()
        if ram_status()["percent"] >= cfg.max_ram_percent:
            raise ResourceLimit("Недостаточно свободной RAM перед light baseline")

    fs = list(features)
    cats = [c for c in fs if c in set(cats_all)]
    params = dict(cfg.search_catboost_params)
    params["iterations"] = cfg.baseline_iterations
    params["random_seed"] = seed
    params["thread_count"] = cfg.thread_count
    params["used_ram_limit"] = cfg.catboost_ram_limit

    model = CatBoostClassifier(**params)
    t0 = time.perf_counter()
    try:
        print(f"Light baseline: {len(fs)} факторов | threads={cfg.thread_count} | RAM={cfg.catboost_ram_limit}")
        model.fit(
            train.loc[:, fs], train[target],
            cat_features=cats,
            eval_set=(valid.loc[:, fs], valid[target]),
            early_stopping_rounds=cfg.baseline_early_stopping_rounds,
            verbose=25,
        )
        pv = model.predict_proba(valid.loc[:, fs])[:, 1]
        elapsed = time.perf_counter() - t0
        gv = gini(valid[target], pv)
        return {
            "method":"all_features_light","repeat":0,"seed":seed,
            "search_seconds":elapsed,"total_seconds":elapsed,
            "search_model_fits":1,"full_validation_fits":0,
            "n_features":len(fs),"light_best_gini":gv,"valid_gini":gv,
            "oos_gini":np.nan,"oos_fit_seconds":np.nan,
            "final_iterations":int(model.get_best_iteration())+1,
            "features":json.dumps(sorted(fs), ensure_ascii=False),
        }
    finally:
        del model
        gc.collect()


def final_oos_once(
    train, valid, oos, target, cats_all, chosen: Dict[str, Any], cfg: Config, seed: int
) -> Dict[str, Any]:
    """
    Единственное обращение к OOS после выбора финального набора по VALID.
    Не создаём pd.concat(TRAIN, VALID): CatBoost получает объединённый набор
    через индексы? Для экономии RAM обучаем финальную модель на TRAIN+VALID
    через временный concat только выбранных колонок, а не всей витрины.
    """
    fs = chosen["features"]
    cats = [c for c in fs if c in set(cats_all)]
    cols = fs + [target]

    full = pd.concat(
        [train.loc[:, cols], valid.loc[:, cols]],
        axis=0,
        ignore_index=True,
        copy=False,
    )

    params = dict(cfg.final_catboost_params)
    params["random_seed"] = seed
    params["thread_count"] = cfg.thread_count
    params["used_ram_limit"] = cfg.catboost_ram_limit
    params["iterations"] = max(1, int(chosen["best_iteration"]) + 1)

    model = CatBoostClassifier(**params)
    t0 = time.perf_counter()
    try:
        model.fit(full.loc[:, fs], full[target], cat_features=cats, verbose=False)
        po = model.predict_proba(oos.loc[:, fs])[:, 1]
        return {
            "oos_gini": gini(oos[target], po),
            "oos_fit_seconds": time.perf_counter() - t0,
            "final_iterations": params["iterations"],
        }
    finally:
        del model
        del full
        gc.collect()


## 4. Вспомогательные функции и методы поиска

In [ ]:
def plateau_update(history: List[float], delta: float, patience: int, min_features_ok: bool) -> bool:
    """
    Plateau считается по лучшему frontier, а не по произвольным последовательным fit.
    Остановка разрешена только после min_features_before_stop.
    """
    if not min_features_ok or len(history) < patience + 1:
        return False
    frontier = np.maximum.accumulate(np.asarray(history, dtype=float))
    gains = np.diff(frontier[-(patience + 1):])
    return bool(np.all(gains < delta))


def _sample_remaining(rng, remaining, n):
    if len(remaining) <= n:
        return list(remaining)
    return rng.choice(remaining, size=n, replace=False).tolist()


def _candidate_blocks(rng, remaining, block_size, n_blocks):
    """Создаёт несколько разных блоков для block-forward шага."""
    remaining = list(remaining)
    if not remaining:
        return []
    b = min(block_size, len(remaining))
    out, seen = [], set()
    attempts = 0
    while len(out) < n_blocks and attempts < n_blocks * 10:
        attempts += 1
        block = tuple(sorted(rng.choice(remaining, size=b, replace=False).tolist()))
        if block not in seen:
            seen.add(block)
            out.append(list(block))
    return out


def greedy_search(ev: Evaluator, features: List[str], cfg: Config, rng) -> Dict[str, Any]:
    """
    Block-forward Greedy.
    В v7 один fit добавлял один фактор, поэтому бюджет 60 fit физически не позволял
    исследовать область 80–200 факторов. Теперь один успешный шаг добавляет блок.
    """
    selected = [f for f in cfg.start_features if f in features]
    history, best_subset, best_gini = [], list(selected), -np.inf

    try:
        while len(selected) < min(cfg.max_features, len(features)):
            remaining = [f for f in features if f not in selected]
            if not remaining:
                break

            block_size = min(
                cfg.additions_per_step,
                cfg.max_features - len(selected),
                len(remaining),
            )
            blocks = _candidate_blocks(
                rng, remaining, block_size, cfg.candidate_pool_size
            )
            if not blocks:
                break

            trials = []
            for block in blocks:
                fs = selected + block
                r = ev.evaluate(
                    fs, "greedy_block",
                    note=f"add_block={len(block)}; n={len(fs)}"
                )
                trials.append((r["valid_gini"], block))

            score, chosen_block = max(trials, key=lambda x: x[0])
            selected.extend(chosen_block)
            history.append(score)

            if score > best_gini:
                best_gini, best_subset = score, list(selected)

            if plateau_update(
                history,
                cfg.delta_gini,
                cfg.plateau_patience,
                len(selected) >= cfg.min_features_before_stop,
            ):
                break
    except (BudgetExceeded, ResourceLimit):
        pass

    if not best_subset:
        fs = _sample_remaining(
            rng, features, min(cfg.additions_per_step, len(features))
        )
        r = ev.evaluate(fs, "greedy_fallback")
        best_subset, best_gini = fs, r["valid_gini"]

    return {
        "features": sorted(best_subset),
        "valid_gini": best_gini,
        "history": history,
    }


def beam_search(ev: Evaluator, features: List[str], cfg: Config, rng) -> Dict[str, Any]:
    """
    Block Beam Search.
    Beam расширяется блоками, поэтому при том же fit-budget может дойти до больших N.
    """
    initial_size = min(cfg.additions_per_step, cfg.max_features, len(features))
    start_blocks = _candidate_blocks(
        rng, features, initial_size, cfg.candidate_pool_size
    )
    beam, history = [], []
    best_subset, best_gini = [], -np.inf

    try:
        for block in start_blocks:
            r = ev.evaluate(block, "beam_start_block", note=f"n={len(block)}")
            beam.append((list(block), r["valid_gini"]))
            if r["valid_gini"] > best_gini:
                best_subset, best_gini = list(block), r["valid_gini"]

        beam = sorted(beam, key=lambda x: x[1], reverse=True)[:cfg.beam_width]
        if beam:
            history.append(beam[0][1])

        while beam and len(beam[0][0]) < min(cfg.max_features, len(features)):
            expanded = {}
            for subset, _ in beam:
                remaining = [f for f in features if f not in subset]
                if not remaining:
                    continue

                block_size = min(
                    cfg.additions_per_step,
                    cfg.max_features - len(subset),
                    len(remaining),
                )
                # При beam_width=2 и pool=6 это 12 fit/уровень.
                blocks = _candidate_blocks(
                    rng, remaining, block_size, cfg.candidate_pool_size
                )

                for block in blocks:
                    new = tuple(sorted(set(subset).union(block)))
                    if len(new) > cfg.max_features or new in expanded:
                        continue
                    r = ev.evaluate(
                        new, "beam_expand_block",
                        note=f"add_block={len(block)}; n={len(new)}"
                    )
                    expanded[new] = r["valid_gini"]

            if not expanded:
                break

            ranked = sorted(
                expanded.items(), key=lambda kv: kv[1], reverse=True
            )[:cfg.beam_width]
            beam = [(list(k), v) for k, v in ranked]
            history.append(beam[0][1])

            if beam[0][1] > best_gini:
                best_subset, best_gini = list(beam[0][0]), beam[0][1]

            if plateau_update(
                history,
                cfg.delta_gini,
                cfg.plateau_patience,
                len(beam[0][0]) >= cfg.min_features_before_stop,
            ):
                break
    except (BudgetExceeded, ResourceLimit):
        pass

    return {
        "features": sorted(best_subset),
        "valid_gini": best_gini,
        "history": history,
    }


def random_search(ev: Evaluator, features: List[str], cfg: Config, rng) -> Dict[str, Any]:
    """Random subset baseline в расширенном диапазоне N."""
    best_subset, best_gini = [], -np.inf
    history = []
    min_n = max(1, min(20, cfg.max_features, len(features)))
    max_n = min(cfg.max_features, len(features))

    try:
        while True:
            n = int(rng.integers(min_n, max_n + 1))
            subset = rng.choice(features, size=n, replace=False).tolist()
            r = ev.evaluate(subset, "random_subset", note=f"n={n}")
            history.append(r["valid_gini"])
            if r["valid_gini"] > best_gini:
                best_subset, best_gini = subset, r["valid_gini"]
    except (BudgetExceeded, ResourceLimit):
        pass

    return {
        "features": sorted(best_subset),
        "valid_gini": best_gini,
        "history": history,
    }


def _make_sampler(method: str, seed: int):
    if optuna is None:
        raise ImportError("Optuna отсутствует: preflight должен был остановить запуск.")

    if method == "tpe":
        return optuna.samplers.TPESampler(
            seed=seed,
            multivariate=True,
            n_startup_trials=10,
        )

    if method == "gp":
        if not hasattr(optuna.samplers, "GPSampler"):
            raise RuntimeError("В установленной Optuna нет GPSampler")
        return optuna.samplers.GPSampler(
            seed=seed,
            n_startup_trials=10,
        )

    if method == "cmaes":
        return optuna.samplers.CmaEsSampler(
            seed=seed,
            n_startup_trials=10,
            warn_independent_sampling=False,
        )

    raise ValueError(method)


def _build_optimizer_blocks(features: List[str], cfg: Config) -> List[List[str]]:
    """
    Search pool уже отсортирован: сначала top importance, затем exploration.
    Делим его на компактные блоки. Optimizer выбирает веса блоков и N,
    а не оптимизирует 270 независимых feature-score параметров.
    Это делает GP/TPE/CMA-ES вычислительно осмысленными.
    """
    max_blocks = min(
        cfg.optimizer_max_blocks,
        int(np.ceil(min(len(features), cfg.max_features) / cfg.optimizer_block_size)),
    )
    usable = min(len(features), max_blocks * cfg.optimizer_block_size)
    pool = list(features[:usable])

    blocks = []
    for i in range(0, len(pool), cfg.optimizer_block_size):
        blocks.append(pool[i:i + cfg.optimizer_block_size])
    return blocks


def optuna_subset_search(
    ev: Evaluator,
    features: List[str],
    cfg: Config,
    method: str,
    rng,
):
    """
    TPE / GP / CMA-ES на одном и том же компактном пространстве.

    Параметры:
    - n_blocks: сколько блоков войдёт в subset;
    - w_i: приоритет каждого блока.

    Итоговый subset = top-n_blocks по w_i.
    При block_size=10 исследуется примерно N=20..200.
    """
    sampler = _make_sampler(method, ev.seed)
    study = optuna.create_study(direction="maximize", sampler=sampler)

    blocks = _build_optimizer_blocks(features, cfg)
    if not blocks:
        raise RuntimeError("Пустой optimizer block pool")

    min_blocks = min(cfg.optimizer_min_blocks, len(blocks))
    max_blocks = min(len(blocks), cfg.optimizer_max_blocks)

    best_subset, best_gini = [], -np.inf
    history = []

    def objective(trial):
        nonlocal best_subset, best_gini

        n_blocks = trial.suggest_int("n_blocks", min_blocks, max_blocks)
        weights = [
            (trial.suggest_float(f"w_{i}", 0.0, 1.0), i)
            for i in range(len(blocks))
        ]
        chosen_idx = [
            i for _, i in sorted(weights, reverse=True)[:n_blocks]
        ]

        subset = []
        for i in chosen_idx:
            subset.extend(blocks[i])
        subset = list(dict.fromkeys(subset))[:cfg.max_features]

        r = ev.evaluate(
            subset,
            f"{method}_blocks",
            note=f"blocks={n_blocks}; n={len(subset)}",
        )
        history.append(r["valid_gini"])

        if r["valid_gini"] > best_gini:
            best_subset = list(subset)
            best_gini = r["valid_gini"]

        return r["valid_gini"]

    try:
        study.optimize(
            objective,
            n_trials=min(cfg.optimizer_trials_per_step, cfg.max_model_fits),
            timeout=cfg.optimizer_timeout_per_step,
            show_progress_bar=False,
            catch=(BudgetExceeded, ResourceLimit),
        )
    except (BudgetExceeded, ResourceLimit):
        pass

    return {
        "features": sorted(best_subset),
        "valid_gini": best_gini,
        "history": history,
    }


def screen_feature_pool(train, valid, features, cats, cfg: Config, seed: int):
    """
    Общий model-based screening для ВСЕХ методов.
    Один лёгкий CatBoost видит факторы совместно, поэтому это не univariate target-filter.
    Exploration-добавка сохраняет часть низкоранговых факторов.
    """
    if not cfg.screening_enabled or len(features) <= cfg.screening_top_n + cfg.screening_random_n:
        return list(features), pd.DataFrame({
            "feature": features,
            "importance": np.nan,
            "source": "all",
        })

    print_ram("Перед screening: ")
    params = dict(cfg.search_catboost_params)
    params["iterations"] = cfg.screening_iterations
    params["thread_count"] = cfg.thread_count
    params["used_ram_limit"] = cfg.catboost_ram_limit
    params["random_seed"] = seed

    model = CatBoostClassifier(**params)
    try:
        model.fit(
            train.loc[:, features], train[cfg.target],
            cat_features=[c for c in features if c in set(cats)],
            eval_set=(valid.loc[:, features], valid[cfg.target]),
            early_stopping_rounds=cfg.search_early_stopping_rounds,
            verbose=False,
        )
        imp = model.get_feature_importance(type="PredictionValuesChange")
    finally:
        del model
        gc.collect()

    ranking = pd.DataFrame({"feature": features, "importance": imp})
    ranking = ranking.sort_values("importance", ascending=False).reset_index(drop=True)

    top = ranking.head(min(cfg.screening_top_n, len(ranking)))["feature"].tolist()
    rest = ranking.loc[~ranking["feature"].isin(top), "feature"].tolist()

    rng = np.random.default_rng(seed)
    random_n = min(cfg.screening_random_n, len(rest))
    explore = rng.choice(rest, size=random_n, replace=False).tolist() if random_n else []

    pool = top + explore
    ranking["source"] = np.where(
        ranking["feature"].isin(top), "top_importance",
        np.where(ranking["feature"].isin(explore), "random_exploration", "outside_pool")
    )
    return pool, ranking


## 5. Диагностика drift и стабильности состава факторов

In [ ]:
def jaccard(a: Sequence[str], b: Sequence[str]) -> float:
    A, B = set(a), set(b)
    return len(A & B) / len(A | B) if (A | B) else 1.0


def population_stability_index(train_s: pd.Series, other_s: pd.Series, bins: int = 10) -> float:
    """Simple PSI diagnostic. Numeric uses TRAIN quantiles; categorical uses TRAIN levels."""
    eps = 1e-6
    try:
        if pd.api.types.is_numeric_dtype(train_s):
            clean = train_s.replace([np.inf, -np.inf], np.nan)
            qs = np.unique(clean.dropna().quantile(np.linspace(0, 1, bins + 1)).values)
            if len(qs) < 3:
                return 0.0
            qs[0], qs[-1] = -np.inf, np.inf
            a = pd.cut(clean, qs, include_lowest=True).value_counts(normalize=True, sort=False)
            b = pd.cut(other_s.replace([np.inf, -np.inf], np.nan), qs, include_lowest=True).value_counts(normalize=True, sort=False)
        else:
            a = train_s.astype("string").fillna("__MISSING__").value_counts(normalize=True)
            b = other_s.astype("string").fillna("__MISSING__").value_counts(normalize=True)
            idx = a.index.union(b.index)
            a, b = a.reindex(idx, fill_value=0), b.reindex(idx, fill_value=0)
        av, bv = np.clip(a.values.astype(float), eps, None), np.clip(b.values.astype(float), eps, None)
        return float(np.sum((bv - av) * np.log(bv / av)))
    except Exception:
        return float("nan")


def drift_report(train: pd.DataFrame, valid: pd.DataFrame, oos: pd.DataFrame, features: Sequence[str]) -> pd.DataFrame:
    rows = []
    for f in features:
        rows.append({
            "feature": f,
            "psi_train_valid": population_stability_index(train[f], valid[f]),
            "psi_train_oos": population_stability_index(train[f], oos[f]),
            "train_missing_rate": float(train[f].isna().mean()),
            "valid_missing_rate": float(valid[f].isna().mean()),
            "oos_missing_rate": float(oos[f].isna().mean()),
            "train_nunique": int(train[f].nunique(dropna=True)),
        })
    return pd.DataFrame(rows)

## 6. Запуск benchmark и формирование отчётов

In [ ]:
def run_search_only(method, repeat, seed, search_train, search_valid, search_features, cats, cfg):
    """Только feature search. FULL/OOS здесь принципиально не вызываются."""
    ev = Evaluator(search_train, search_valid, cfg.target, cats, cfg, method, repeat, seed)
    rng = np.random.default_rng(seed)
    t0 = time.perf_counter()

    if method == "greedy":
        result = greedy_search(ev, search_features, cfg, rng)
    elif method == "beam":
        result = beam_search(ev, search_features, cfg, rng)
    elif method == "random":
        result = random_search(ev, search_features, cfg, rng)
    elif method in {"tpe", "gp", "cmaes"}:
        result = optuna_subset_search(ev, search_features, cfg, method, rng)
    else:
        raise ValueError(f"Неизвестный метод: {method}")

    traj = pd.DataFrame(ev.rows)
    summary = {
        "method": method,
        "repeat": repeat,
        "seed": seed,
        "search_seconds": time.perf_counter() - t0,
        "search_model_fits": ev.fit_count,
        "n_features": len(result["features"]),
        "search_best_gini": result["valid_gini"],
        "features": json.dumps(result["features"], ensure_ascii=False),
    }
    del ev
    gc.collect()
    return summary, traj


def candidate_sets_from_confirmation(confirm_runs, confirm_traj, method, k=3):
    """Top-K уникальных наборов метода по confirmation SEARCH Gini."""
    candidates = []
    if confirm_traj is not None and not confirm_traj.empty:
        t = confirm_traj[confirm_traj["method"] == method].sort_values(
            "valid_gini", ascending=False
        )
        for _, r in t.iterrows():
            fs = tuple(sorted(json.loads(r["features"])))
            if fs and fs not in [tuple(x) for x in candidates]:
                candidates.append(list(fs))
            if len(candidates) >= k:
                break

    if len(candidates) < k:
        rr = confirm_runs[confirm_runs["method"] == method].sort_values(
            "search_best_gini", ascending=False
        )
        for _, r in rr.iterrows():
            fs = tuple(sorted(json.loads(r["features"])))
            if fs and fs not in [tuple(x) for x in candidates]:
                candidates.append(list(fs))
            if len(candidates) >= k:
                break
    return candidates


def evaluate_finalists_full_and_oos(
    methods, confirm_runs, confirm_traj,
    train, valid, oos, cats, cfg
):
    """
    FULL VALID: top-K уникальных наборов каждого метода.
    OOS: только лучший FULL-набор каждого метода.
    """
    full_rows, final_rows = [], []

    for method in methods:
        print(f"\\n[FULL] method={method}")
        candidates = candidate_sets_from_confirmation(
            confirm_runs, confirm_traj, method, cfg.full_candidates_per_method
        )
        if not candidates:
            warnings.warn(f"{method}: нет кандидатов для FULL")
            continue

        method_full = []
        seed = cfg.confirmation_seeds[0]

        for rank, fs in enumerate(candidates, 1):
            print(f"  FULL candidate {rank}/{len(candidates)} | N={len(fs)}")
            r = fit_full_on_valid(train, valid, cfg.target, cats, fs, cfg, seed)
            r.update({"method": method, "candidate_rank": rank})
            method_full.append(r)
            full_rows.append(r)
            pd.DataFrame(full_rows).to_csv(
                Path(cfg.output_dir) / "full_validation_checkpoint.csv", index=False
            )
            gc.collect()

        chosen = max(method_full, key=lambda x: x["full_valid_gini"])
        print(
            f"  BEST FULL | N={chosen['n_features']} | "
            f"VALID Gini={chosen['full_valid_gini']:.5f}"
        )

        # OOS только один раз на метод.
        oos_r = final_oos_once(
            train, valid, oos, cfg.target, cats, chosen, cfg, seed
        )
        final_rows.append({
            "method": method,
            "n_features": chosen["n_features"],
            "valid_gini": chosen["full_valid_gini"],
            **oos_r,
            "features": json.dumps(chosen["features"], ensure_ascii=False),
        })
        pd.DataFrame(final_rows).to_csv(
            Path(cfg.output_dir) / "finalists_checkpoint.csv", index=False
        )
        gc.collect()

    return pd.DataFrame(full_rows), pd.DataFrame(final_rows)


def summarize_confirmation(confirm_runs: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for method, g in confirm_runs.groupby("method"):
        sets = [json.loads(x) for x in g["features"]]
        jac = [
            jaccard(sets[i], sets[j])
            for i in range(len(sets))
            for j in range(i + 1, len(sets))
        ]
        rows.append({
            "method": method,
            "runs": len(g),
            "search_gini_mean": g.search_best_gini.mean(),
            "search_gini_std": g.search_best_gini.std(ddof=1) if len(g) > 1 else np.nan,
            "n_features_mean": g.n_features.mean(),
            "search_seconds_mean": g.search_seconds.mean(),
            "feature_jaccard_mean": float(np.mean(jac)) if jac else np.nan,
        })
    return pd.DataFrame(rows).sort_values("search_gini_mean", ascending=False)


In [ ]:
out = Path(cfg.output_dir)
out.mkdir(parents=True, exist_ok=True)

validate_experiment_protocol(cfg)
validate_search_dependencies(cfg)

print("Загрузка датасетов...")
train = load_table(cfg.train)
valid = load_table(cfg.valid)
oos = load_table(cfg.oos)
print_ram("После загрузки: ")

train[cfg.target] = validate_binary_target(train[cfg.target], "TRAIN")
valid[cfg.target] = validate_binary_target(valid[cfg.target], "VALID")
oos[cfg.target] = validate_binary_target(oos[cfg.target], "OOS")

train, valid, oos, features, cats, technical, type_audit, generated_dates = technical_filter(
    train, valid, oos, cfg
)
gc.collect()
print_ram("После preprocessing: ")

if not features:
    raise ValueError("После preprocessing не осталось факторов")

print(f"TRAIN={len(train):,}; VALID={len(valid):,}; OOS={len(oos):,}")
print(f"Факторов={len(features)}; категориальных={len(cats)}")

type_audit.to_csv(out / "type_audit.csv", index=False)
technical.to_csv(out / "technical_feature_audit.csv", index=False)

# ---------- Два независимых SEARCH-уровня ----------
t_train = stratified_binary_sample(
    train, cfg.target, cfg.tournament_train_rows, cfg.search_sample_seed
)
t_valid = stratified_binary_sample(
    valid, cfg.target, cfg.tournament_valid_rows, cfg.search_sample_seed + 1
)
c_train = stratified_binary_sample(
    train, cfg.target, cfg.confirmation_train_rows, cfg.search_sample_seed + 10
)
c_valid = stratified_binary_sample(
    valid, cfg.target, cfg.confirmation_valid_rows, cfg.search_sample_seed + 11
)

print(
    f"TOURNAMENT: TRAIN={len(t_train):,}, VALID={len(t_valid):,} | "
    f"CONFIRMATION: TRAIN={len(c_train):,}, VALID={len(c_valid):,}"
)
print_ram("После SEARCH samples: ")

# Drift не участвует в selection.
if cfg.calculate_drift:
    print("Расчёт drift...")
    drift = drift_report(train, valid, oos, features)
    drift.to_csv(out / "drift_report.csv", index=False)
else:
    print("Drift пропущен (calculate_drift=False).")

# ---------- Единый screening на дешёвом Tournament sample ----------
print("\\n=== ОБЩИЙ MODEL-BASED SCREENING ===")
screen_cfg = replace(
    cfg,
    search_iterations=min(cfg.tournament_iterations, cfg.screening_iterations),
    max_model_fits=cfg.tournament_fits,
)
search_features, screening = screen_feature_pool(
    t_train, t_valid, features, cats, screen_cfg, cfg.tournament_seed
)
screening.to_csv(out / "screening_importance.csv", index=False)
print(f"Поисковый пул: {len(search_features)} из {len(features)}")
print_ram("После screening: ")

# ---------- STAGE 1: TOURNAMENT, ВСЕ 6 МЕТОДОВ ----------
print("\\n================ TOURNAMENT: ВСЕ МЕТОДЫ ================")
tournament_runs, tournament_traj = [], []

t_cfg = replace(
    cfg,
    max_model_fits=cfg.tournament_fits,
    search_iterations=cfg.tournament_iterations,
    max_wall_seconds=min(cfg.max_wall_seconds, 1200),
)

for method in cfg.methods:
    key = (method, cfg.tournament_seed)
    print(f"\\n[TOURNAMENT] {method} | seed={cfg.tournament_seed}")
    try:
        summary, traj = run_search_only(
            method, 0, cfg.tournament_seed,
            t_train, t_valid, search_features, cats, t_cfg
        )
        tournament_runs.append(summary)
        tournament_traj.append(traj)
        pd.DataFrame(tournament_runs).to_csv(out / "tournament_runs.csv", index=False)
        pd.concat(tournament_traj, ignore_index=True).to_csv(
            out / "tournament_trajectories.csv", index=False
        )
    except Exception as e:
        with (out / "errors.jsonl").open("a", encoding="utf-8") as fh:
            fh.write(json.dumps({
                "stage": "tournament", "method": method,
                "seed": cfg.tournament_seed, "error": repr(e)
            }, ensure_ascii=False) + "\\n")
        warnings.warn(f"Tournament {method}: {e}")
    finally:
        gc.collect()
        print_ram("После tournament method: ")

tournament_df = pd.DataFrame(tournament_runs)
if tournament_df.empty:
    raise RuntimeError("Tournament не завершил ни одного метода")

# Все 6 должны быть реально проверены.
missing_methods = sorted(set(cfg.methods) - set(tournament_df["method"]))
if missing_methods:
    raise RuntimeError(
        "Tournament не проверил все методы. Не запускаем confirmation. "
        f"Не завершились: {missing_methods}. См. errors.jsonl"
    )

tournament_df = tournament_df.sort_values("search_best_gini", ascending=False)
tournament_df.to_csv(out / "tournament_ranking.csv", index=False)
display(tournament_df)

finalist_methods = tournament_df.head(cfg.confirmation_top_methods)["method"].tolist()
print(f"\\nФиналисты Tournament: {finalist_methods}")

# ---------- STAGE 2: CONFIRMATION, TOP-3 × 3 SEED ----------
print("\\n================ CONFIRMATION ================")
confirm_runs, confirm_traj = [], []

c_cfg = replace(
    cfg,
    max_model_fits=cfg.confirmation_fits,
    search_iterations=cfg.confirmation_iterations,
    max_wall_seconds=min(cfg.max_wall_seconds, 1800),
)

# Resume на уровне законченного method×seed.
confirm_checkpoint = out / "confirmation_runs.csv"
done = set()
if cfg.resume and confirm_checkpoint.exists():
    old = pd.read_csv(confirm_checkpoint)
    for _, r in old.iterrows():
        done.add((str(r["method"]), int(r["seed"])))
        confirm_runs.append(r.to_dict())
    print(f"Resume: найдено {len(done)} завершённых confirmation run.")

for method in finalist_methods:
    for repeat, seed in enumerate(cfg.confirmation_seeds):
        if (method, seed) in done:
            print(f"[RESUME] пропуск {method}, seed={seed}: уже завершён")
            continue

        print(f"\\n[CONFIRM] {method} | repeat={repeat} | seed={seed}")
        try:
            summary, traj = run_search_only(
                method, repeat, seed,
                c_train, c_valid, search_features, cats, c_cfg
            )
            confirm_runs.append(summary)
            confirm_traj.append(traj)

            pd.DataFrame(confirm_runs).to_csv(
                out / "confirmation_runs.csv", index=False
            )
            if confirm_traj:
                # Новые траектории текущей сессии; fit_journal хранит все fit.
                pd.concat(confirm_traj, ignore_index=True).to_csv(
                    out / "confirmation_trajectories_session.csv", index=False
                )
        except Exception as e:
            with (out / "errors.jsonl").open("a", encoding="utf-8") as fh:
                fh.write(json.dumps({
                    "stage": "confirmation", "method": method,
                    "repeat": repeat, "seed": seed, "error": repr(e)
                }, ensure_ascii=False) + "\\n")
            warnings.warn(f"Confirmation {method}, seed={seed}: {e}")
        finally:
            gc.collect()
            print_ram("После confirmation run: ")

confirm_df = pd.DataFrame(confirm_runs)
if confirm_df.empty:
    raise RuntimeError("Confirmation не дал результатов")

confirm_summary = summarize_confirmation(confirm_df)
confirm_summary.to_csv(out / "confirmation_summary.csv", index=False)
display(confirm_summary)

# Для shortlist используем fit_journal, чтобы не терять траектории после resume.
journal_path = out / "fit_journal.jsonl"
journal_rows = []
if journal_path.exists():
    with journal_path.open("r", encoding="utf-8") as fh:
        for line in fh:
            try:
                journal_rows.append(json.loads(line))
            except Exception:
                pass
all_traj = pd.DataFrame(journal_rows)

# ---------- STAGE 3/4: FULL VALID + OOS ----------
print("\\n================ FULL VALID + OOS ================")
full_df, finalists_df = evaluate_finalists_full_and_oos(
    finalist_methods, confirm_df, all_traj,
    train, valid, oos, cats, cfg
)

full_df.to_csv(out / "full_validation_candidates.csv", index=False)
finalists_df.to_csv(out / "finalists_oos.csv", index=False)

print("\\n=== ИТОГОВЫЕ ФИНАЛИСТЫ ===")
display(finalists_df.sort_values("oos_gini", ascending=False))
print(f"\\nРезультаты: {out.resolve()}")


## 7. Протокол v9

### Stage 1 — Tournament
Все шесть методов обязательно получают одинаковый дешёвый бюджет:
- 150k TRAIN;
- 50k VALID;
- 1 seed;
- максимум 30 CatBoost-fit;
- 90 итераций search-модели.

Если хотя бы один из шести методов не завершился, Confirmation **не запускается**. Это не позволяет снова получить «сравнение» без TPE/GP/CMA-ES.

### Stage 2 — Confirmation
Автоматически выбираются top-3 метода Tournament. Каждый запускается:
- на 300k TRAIN + 100k VALID;
- по seed `42 / 2026 / 3407`;
- максимум 30 fit;
- 120 итераций.

### Stage 3 — FULL VALID
Для каждого метода-финалиста максимум 3 уникальных набора переоцениваются на полном TRAIN/VALID.

### Stage 4 — OOS
Только лучший по FULL VALID набор каждого метода один раз проверяется на OOS.

### Устойчивость
`thread_count=8`, heartbeat после каждого fit, `fit_journal.jsonl` после каждого fit и resume завершённых `method × seed` в Confirmation. Модели CatBoost после оценки удаляются.


In [ ]:
# Быстрый просмотр результатов v9
if 'tournament_df' in globals():
    display(tournament_df)
if 'confirm_summary' in globals():
    display(confirm_summary)
if 'finalists_df' in globals() and not finalists_df.empty:
    display(finalists_df.sort_values("oos_gini", ascending=False))


In [ ]:
def build_gini_frontier(trajectory_df: pd.DataFrame) -> pd.DataFrame:
    """
    Диагностический frontier для v5:
    для каждого количества факторов показывает лучший LIGHT VALID Gini,
    который реально был найден поиском.
    """
    if trajectory_df.empty:
        return pd.DataFrame()
    return (
        trajectory_df[["n_features", "valid_gini"]]
        .dropna()
        .groupby("n_features", as_index=False)["valid_gini"]
        .max()
        .sort_values("n_features")
        .reset_index(drop=True)
    )

frontier = build_gini_frontier(traj_df)
if not frontier.empty:
    display(frontier)
